In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_core.prompts import PromptTemplate

In [2]:
llm = OllamaLLM(model="llama3.2")

In [3]:
file_path = "nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))
#print(f"{docs[0].page_content[:200]}\n")
#print(docs[0].metadata)

107


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

516

In [ ]:
embeddings = OllamaEmbeddings(
    model="llama3.2",
)
vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=all_splits)

In [ ]:
#Use three sentences maximum and keep the answer concise.
prompt = ChatPromptTemplate.from_template("""
You are a specialist on the domain for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. 

Question: {question} 
Context: {context} 
Answer:
""")
prompt

In [ ]:
#question 
question = {"question": "How many distribution centers does Nike have in the US?"}

#retrieve
retrieved_docs = vector_store.similarity_search(question)
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

#create input
input = {"question": question['question'], "context": docs_content}


messages = prompt.invoke(input)
#chain = prompt | llm
#chain.invoke(question)

answer = llm.invoke(prompt)



In [ ]:
answer.content

Similarity search 

In [ ]:
#results = await vector_store.asimilarity_search("When was Nike incorporated?")
#print(results[0])

In [ ]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Retrieving

In [ ]:
# Testes de embed
'''
# You can embed single texts or documents with embed_query:
# single_vector = embeddings.embed_query(text)
# print(str(single_vector)[:100])  # Show the first 100 characters of the vector

text2 = (
    "LangGraph is a library for building stateful, multi-actor applications with LLMs"
)
two_vectors = embeddings.embed_documents([text, text2])
for vector in two_vectors:
    print(str(vector)[:100])  # Show the first 100 characters of the vector'''

# Create a vector store with a sample text
text = "LangChain is the framework for building context-aware reasoning applications"

vectorstore = InMemoryVectorStore.from_texts(
    [text, "a framework is boring", "data is necessary for any framework", "a framework can be dull", "a long text can cause more implication on finding a specific word such as a framework"],
    embedding=embeddings,
)

# Use the vectorstore as a retriever
retriever = vectorstore.as_retriever()

# Retrieve the most similar text
retrieved_documents = retriever.invoke("What is LangChain?")

# show the retrieved document's content
retrieved_documents[0].page_content

In [ ]:
retrieved_documents = retriever.invoke("framework")

# show the retrieved document's content
retrieved_documents[0].page_content

In [ ]:
retrieved_documents